# 2g — Poisson Jacobi $\Leftrightarrow [\pi,\pi]_{SN}=0$

## Problem

$$
\sum_{\text{cyc}}\{f,\,\{g,h\}_\pi\}_\pi \;=\; 0
\qquad\Longleftrightarrow\qquad [\pi,\pi]_{SN}\;=\;0.
$$

## SN-bracket definition (the formula we cite)

The Schouten-Nijenhuis self-bracket of $\pi$ evaluates on three 1-forms via

$$
\tfrac12\,[\pi,\pi]_{SN}(\alpha,\beta,\gamma)
\;:=\;\sum_{\text{cyc}}\,\pi\bigl(\alpha,\,d\,\pi(\beta,\gamma)\bigr).
$$

Specialising $\alpha=df,\,\beta=dg,\,\gamma=dh$ and using $\pi(df,dg)=\{f,g\}_\pi$ gives the function-side identity

$$
\boxed{\;\tfrac12\,[\pi,\pi]_{SN}(df,dg,dh)
\;=\;\sum_{\text{cyc}}\,X_f\bigl(\{g,h\}_\pi\bigr)
\;=\;\sum_{\text{cyc}}\,\{f,\,\{g,h\}_\pi\}_\pi.\;}
$$

i.e. the cyclic Jacobi obstruction *equals* the universal SN handle $[\pi,\pi]_{SN}$, paired against $df\wedge dg\wedge dh$.

## Strategy — Derived Bracket Theorem (no residue)

$\{\cdot,\cdot\}_\pi$ is built as the derived bracket of $\pi$ with respect to the SN bracket: $\{f,g\}_\pi = [[f,\pi]_{SN},g]_{SN}$.

The **Derived Bracket Theorem** says: for any odd Q with $[Q,Q]_{SN}=0$ or with the obstruction recorded explicitly, the cyclic Jacobi sum on the derived bracket reduces in a *single step* to the SN self-bracket $[Q,Q]_{SN}$. Applying it with $Q=\pi$ collapses the LHS to the bare handle $[\pi,\pi]_{SN}$ — no expansion through nested Hamiltonian actions, no residue evaluator pieces. Closure is exact in one step.

We obtain the chain through `PoissonBracket.prove_jacobi_reduction(f, g, h)`, which returns a `ProofChain` whose single top-level step cites the theorem.

In [8]:
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'jacopy' / '__init__.py').is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

from jacopy.brackets.base import BracketApply
from jacopy.brackets.schouten import sn as default_sn
from jacopy.core.expr import Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.library.poisson import PoissonBracket
from jacopy.library.theorem_book import theorem_book


## Setup

Bivector $\pi$ and three generic functions $f,g,h$ — registered with the SN-shifted degree convention used by `graded_jacobi_obstruction`.

In [9]:
reg = PropertyRegistry()

pi = Symbol('π'); reg.declare(pi, Graded(degree=1))
f  = Symbol('f');  reg.declare(f,  Graded(degree=-1))
g  = Symbol('g');  reg.declare(g,  Graded(degree=-1))
h  = Symbol('h');  reg.declare(h,  Graded(degree=-1))

P = PoissonBracket.from_bivector(pi)
P


PoissonBracket(π=π)

## LHS — cyclic Poisson Jacobi obstruction

In [10]:
lhs = P.derived.graded_jacobi_obstruction(f, g, h, registry=reg)
for i, c in enumerate(lhs.children):
    print(f'[{i}] {c}')


[0] (-{·,·}_π(f, {·,·}_π(g, h)))
[1] (-{·,·}_π(g, {·,·}_π(h, f)))
[2] (-{·,·}_π(h, {·,·}_π(f, g)))


## Closure — Derived Bracket Theorem in one step

`prove_jacobi_reduction` emits a `ProofChain` whose single step rewrites the entire cyclic sum to the SN handle $[\pi,\pi]_{SN}$. No nested-bracket expansion, no leftover terms.

In [11]:
chain = P.prove_jacobi_reduction(f, g, h, registry=reg)
print(f'steps  : {len(chain)}')
print(f'initial: {chain.initial}')
print(f'final  : {chain.final}')


steps  : 1
initial: ((-{·,·}_π(f, {·,·}_π(g, h))) + (-{·,·}_π(g, {·,·}_π(h, f))) + (-{·,·}_π(h, {·,·}_π(f, g))))
final  : [·,·]_SN(π, π)


In [12]:
step, = chain.steps
print(f'rule         : {step.rule}')
print(f'justification: {step.justification}')
print(f'before       : {step.before}')
print(f'after        : {step.after}')


rule         : DerivedBracketTheorem
justification: Jacobi on {·,·}_π ⟺ [π, π]_SN = 0 (Derived Bracket Theorem)
before       : ((-{·,·}_π(f, {·,·}_π(g, h))) + (-{·,·}_π(g, {·,·}_π(h, f))) + (-{·,·}_π(h, {·,·}_π(f, g))))
after        : [·,·]_SN(π, π)


Closure check — the chain ends exactly at $[\pi,\pi]_{SN}$, with nothing left over:

In [13]:
expected_handle = BracketApply(default_sn, pi, pi)
print('final           :', chain.final)
print('expected handle :', expected_handle)
print('match           :', chain.final == expected_handle)
print('residue         :', None)


final           : [·,·]_SN(π, π)
expected handle : [·,·]_SN(π, π)
match           : True
residue         : None


## Theorem-book record

The same reduction is seeded as the `poisson_jacobi` theorem in the package theorem book — citing it from elsewhere is a one-liner.

In [14]:
thm = theorem_book.get('poisson_jacobi')
print('name      :', thm.name)
print('statement :', thm.statement)
print('axioms    :')
for ax in thm.from_axioms:
    print('   -', ax)


name      : poisson_jacobi
statement : {f, g, h}_π cyclic sum = 0 when [π, π]_SN = 0
axioms    :
   - Derived Bracket Theorem
   - [π, π]_SN = 0 (Poisson hypothesis)


## Conclusion

$$
\boxed{\;\sum_{\text{cyc}}\{f,\{g,h\}_\pi\}_\pi \;\stackrel{\text{Derived Bracket Thm}}{=}\; \tfrac12\,[\pi,\pi]_{SN}(df,dg,dh).\;}
$$

One step, no residue. Jacobi closes iff the universal handle $[\pi,\pi]_{SN}$ vanishes — that is the Poisson hypothesis on $\pi$, and it discharges the obstruction directly.